In [1]:
%config IPCompleter.use_jedi = False
import import_ipynb

In [2]:
from pg_03_01_index import text_search, semantic_search, combined_search

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/474 [00:00<?, ?it/s]

In [9]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

In [50]:
SYSTEM_PROMPT="""
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers and include the reference by citing the filename of the source material.
If the search doesn't return relevant results, let the user know and provide general guidance.

"""

In [51]:
def get_agent(model=None, tools: list = []) -> Agent:
    if model is None:
        model = OpenAIChatModel(
            model_name="qwen2.5:0.5b",
            provider=OllamaProvider(base_url='http://localhost:11434/v1')
        )
    return Agent(
        name="FAQ",
        system_prompt=SYSTEM_PROMPT,
        tools=tools,
        model=model
    )

In [30]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

In [46]:
def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "source": source,
        "messages": dict_messages,
    }

In [47]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)
    print(entry)    
    ts = entry['messages'][-1]['timestamp']
    if isinstance(ts, datetime):
        ts_obj = ts
    elif isinstance(ts, str):
        ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    else:
        raise TypeError(f"Unexpected timestamp type: {type(ts)}")
        
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")

    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

In [52]:
agent = get_agent(tools=[text_search])

In [54]:
question = "Who is the creator of typer?"
result = await agent.run(user_prompt=question)

In [55]:
print(result.output)
print("---")
log_interaction_to_file(agent, result.new_messages())

According to the FAQ, Typer was created by Sebastián Ramírez (@sebastiandramirez) and Sebastián is co-founder. You can find basic information about PyTorch's creator on their official documentation: 

[TensorFlow](https://www.tensorflow.org/) has been heavily influenced by PyTorch, which can be seen from PyTorch's code structure.

The company "fastai", which started in 2018, specializes in machine learning and provides resources to learn about the field. The Python library "Pytorch" is often recommended for beginners or as an example of how to use a popular deep learning model (e.g., ImageNet).
---
{'agent_name': 'FAQ', 'system_prompt': [], 'provider': 'ollama', 'model': 'qwen2.5:0.5b', 'tools': ['text_search'], 'source': 'user', 'messages': [{'parts': [{'content': "\nYou are a helpful assistant for a course. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provid

WindowsPath('logs/FAQ_20260406_181514_1bbd30.json')

In [45]:
result.new_messages()

[ModelRequest(parts=[SystemPromptPart(content="\nYou are a helpful assistant for a course. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provide accurate answers.\nIf the search doesn't return relevant results, let the user know and provide general guidance.\n\n", timestamp=datetime.datetime(2026, 4, 6, 18, 1, 24, 402691, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Who is the creator of typer?', timestamp=datetime.datetime(2026, 4, 6, 18, 1, 24, 402696, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 6, 18, 1, 24, 402902, tzinfo=datetime.timezone.utc), run_id='019d63f4-dab1-72a7-8a15-43be58d293ee'),
 ModelResponse(parts=[ToolCallPart(tool_name='text_search', args='{"query":"TyPer creators"}', tool_call_id='call_2fobfi4y')], usage=RequestUsage(input_tokens=253, output_tokens=22), model_name='qwen2.5:0.5b', timestamp=datetime.dat